# Learning a CMB-like Field as a Neural Field

In this notebook we'll teach a multi-layer perceptron (MLP) to represent a 2D CMB-like temperature field as a *continuous* scalar function — a function

$$ f_\theta: (x, y) \mapsto T $$

mapping any sky coordinate $(x, y)$ to a temperature value $T$. After training we can evaluate $f_\theta$ at any coordinate, including positions between pixels — an effectively infinite-resolution representation of the field.

We'll build up in two acts: a warm-up that grounds the PyTorch training pipeline in something visualizable, then a 2D CMB-like field.

In [ ]:
# Auto-reload imported modules (dataset.py, model.py) on file change so we
# can edit them without restarting the kernel.
%load_ext autoreload
%autoreload 2
%matplotlib inline

## 1. Warm-up

Two warm-ups before M31. First we'll watch gradient descent on a 2-parameter problem so we can actually *see* the loss surface and the optimizer's path. Then we'll run the same pipeline on a 1D function with a small MLP.

In both, knobs sit at the top of each code cell — **re-run with different values** and see what changes.

### 1.1 Gradient descent, visualized

With only two parameters, the loss surface is a 2D function we can plot directly, and the optimizer's trajectory is a path on that surface. Here we fit a line $y = a\,x + b$ to noisy linear data, evaluate the MSE loss on a grid of $(a, b)$ values, and run SGD starting from a chosen initialization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Knobs ---
init_a, init_b = 0.0, 0.0    # starting point in (a, b) space
lr_2d          = 0.5
n_steps_2d     = 50
# ---

# Noisy linear data: true (a, b) = (3.0, 0.5)
torch.manual_seed(0)
true_a, true_b = 3.0, 0.5
x2d = torch.linspace(0, 1, 64, device=device).unsqueeze(1)
y2d = true_a * x2d + true_b + 0.2 * torch.randn_like(x2d)

def loss_fn(a, b):
    return ((a * x2d + b - y2d) ** 2).mean()

# Loss on a (a, b) grid — vectorized in closed form for speed.
mean_x  = x2d.mean(); mean_x2 = (x2d ** 2).mean()
mean_y  = y2d.mean(); mean_y2 = (y2d ** 2).mean()
mean_xy = (x2d * y2d).mean()

a_grid = torch.linspace(-1, 6, 100, device=device)
b_grid = torch.linspace(-2, 3, 100, device=device)
A, B = torch.meshgrid(a_grid, b_grid, indexing="ij")
loss_grid = (A ** 2 * mean_x2 + 2 * A * B * mean_x - 2 * A * mean_xy
             + B ** 2 - 2 * B * mean_y + mean_y2)

# SGD trajectory on (a, b).
a = torch.tensor(init_a, device=device, requires_grad=True)
b = torch.tensor(init_b, device=device, requires_grad=True)
opt = torch.optim.SGD([a, b], lr=lr_2d)

traj = [(a.item(), b.item())]
for step in range(n_steps_2d):
    opt.zero_grad()
    loss_fn(a, b).backward()
    opt.step()
    traj.append((a.item(), b.item()))
traj = np.array(traj)

print(f"Final (a, b) = ({a.item():.3f}, {b.item():.3f}); true = ({true_a}, {true_b})")

In [ ]:
A_cpu   = A.cpu().numpy()
B_cpu   = B.cpu().numpy()
logloss = loss_grid.cpu().log().numpy()

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(A_cpu, B_cpu, logloss, levels=30, cmap="viridis")
fig.colorbar(cf, ax=ax, label="log MSE loss")
ax.contour(A_cpu, B_cpu, logloss, levels=10, colors="white", alpha=0.3, linewidths=0.5)
ax.plot(traj[:, 0], traj[:, 1], "o-", color="red", markersize=3, lw=1, label="SGD trajectory")
ax.plot(traj[0, 0], traj[0, 1], "*", color="yellow", markersize=18, label="start", zorder=5)
ax.plot(true_a, true_b, "+", color="white", markersize=15, mew=2, label="true min", zorder=5)
ax.set_xlabel("a (slope)"); ax.set_ylabel("b (intercept)")
ax.set_title(f"Loss landscape and SGD trajectory ({n_steps_2d} steps, lr={lr_2d})")
ax.legend()
plt.tight_layout()
plt.show()

#### Try these

Re-run the setup cell with these tweaks and watch the trajectory change:

- **Different start.** Try `init_a, init_b = 5, -1.5` or `(-1, 2.5)`. The bowl is convex, so SGD still finds the same minimum — but the path looks very different.
- **Bigger learning rate.** Set `lr_2d = 1.5`. Does the trajectory zig-zag across the valley before settling?
- **Way too big.** Set `lr_2d = 3.0`. Does SGD overshoot and diverge?
- **Tiny learning rate.** Set `lr_2d = 0.05`. How many steps does it take now to reach the bottom?

Notice the valley is elongated — the loss is much more sensitive to `a` than `b`. A single fixed step size has to be small enough not to overshoot along the steep direction, which makes progress along the shallow one painfully slow. This is exactly the problem **adaptive optimizers** like Adam (what we'll use for the MLP next) are designed to fix — they pick a per-parameter step size automatically.

### 1.2 The PyTorch pipeline on a 1D function

Now scale up: same loop (`zero_grad → forward → loss → backward → step`), but the model is a small MLP instead of two scalars, and we use Adam instead of SGD. The loss surface lives in hundreds of dimensions — we can't plot it anymore — but the principle is the same.

In [ ]:
# --- Target function ---
def target_fn(x):
    return torch.sin(2 * np.pi * x) + 0.3 * torch.cos(6 * np.pi * x)
# ---

n_samples = 256
x = torch.linspace(0, 1, n_samples, device=device).unsqueeze(1)   # (N, 1)
y = target_fn(x) + 0.05 * torch.randn_like(x)                     # (N, 1) noisy targets

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x.cpu(), y.cpu(), ".", alpha=0.5, label="data")
ax.plot(x.cpu(), target_fn(x).cpu(), "-", label="truth")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Model and training knobs ---
hidden_width  = 32
hidden_layers = 3
lr            = 1e-2
n_steps       = 1000
# ---

# A tiny MLP, defined inline so the architecture is fully visible.
layers = [nn.Linear(1, hidden_width), nn.ReLU()]
for _ in range(hidden_layers - 1):
    layers += [nn.Linear(hidden_width, hidden_width), nn.ReLU()]
layers += [nn.Linear(hidden_width, 1)]
model = nn.Sequential(*layers).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Snapshot the prediction at initialization for the before/after plot.
with torch.no_grad():
    y_pred_init = model(x).cpu()

losses = []
for step in range(n_steps):
    optimizer.zero_grad()
    y_pred = model(x)
    loss = ((y_pred - y) ** 2).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (step + 1) % (n_steps // 10) == 0:
        print(f"step {step + 1:>5} | loss {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    y_pred_final = model(x).cpu()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x.cpu(), y.cpu(), ".", alpha=0.4, label="data")
axes[0].plot(x.cpu(), y_pred_init,  "-", label="prediction (init)")
axes[0].plot(x.cpu(), y_pred_final, "-", label="prediction (trained)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("Fit"); axes[0].legend()

axes[1].semilogy(losses)
axes[1].set_xlabel("step"); axes[1].set_ylabel("MSE loss")
axes[1].set_title("Training loss")

plt.tight_layout()
plt.show()

#### Try these

Re-run the training cell with each tweak and watch what changes:

- **Bigger model.** Set `hidden_width = 128` or `hidden_layers = 6`. Does the fit get visibly better?
- **Smaller model.** Set `hidden_width = 4`. Can the MLP represent the wiggly target at all?
- **Higher learning rate.** Set `lr = 1e-1`. Does training still converge, or does the loss blow up?
- **More steps.** Set `n_steps = 5000`. Does the loss keep decreasing or plateau?
- **Harder target.** Add a higher-frequency term to `target_fn`, e.g. `+ 0.2 * torch.sin(20 * np.pi * x)`. Does the MLP capture it?

That last one previews a key point we'll hit in M31: **MLPs struggle with high-frequency content.** Hold onto that observation.

## 2. The dataset and the problem

### A CMB-like Gaussian random field

Our target is a 2D Gaussian random field with a power-law spectrum $P(k) \propto k^\alpha$ — the same statistical model that describes the cosmic microwave background's temperature anisotropies and, in the linear regime, the large-scale structure of the universe. Each Fourier mode is given a random complex amplitude with magnitude $\propto k^{\alpha/2}$, then inverse-FFT'd into real space; with $\alpha \approx -2.5$ we get smooth multi-scale structure that's well-conditioned for neural-field fitting — no point sources, no extreme dynamic range, no foreground contamination.

The script in `scripts/gen_field.py` generates the field and saves it to `data/cmb/cmb_1k.npy`.

In [ ]:
from dataset import DEFAULT_PATH

data = np.load(DEFAULT_PATH)
print(f"loaded: {DEFAULT_PATH}")
print(f"shape: {data.shape}, dtype: {data.dtype}")
print(f"min: {data.min():.3g}, max: {data.max():.3g}, "
      f"mean: {data.mean():.3g}, median: {np.median(data):.3g}")
print(f"pixels: {data.size:,} (this is how many training samples we have)")

In [ ]:
# The field is Gaussian by construction — symmetric around zero, no heavy
# tails. Min-max normalization to [-1, 1] is all the preprocessing it needs.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(data.flatten(), bins=100)
ax.set_xlabel("field value")
ax.set_ylabel("count")
ax.set_title("CMB-like field value distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Symmetric diverging colormap for a zero-centered field (CMB convention).
vmax = np.abs(data).max()
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(data, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_title("CMB-like Gaussian random field (synthetic)")
ax.set_xlabel("pixel x"); ax.set_ylabel("pixel y")
plt.tight_layout()
plt.show()

### The problem: continuous field representation

The data is discrete — a regular grid of pixels — but the underlying field is continuous. We want to recover that continuous function:

$$ f: \mathbb{R}^2 \to \mathbb{R}, \quad (x, y) \mapsto T(x, y) $$

and we'll learn it with an MLP — same pipeline as the warm-up, just 2D inputs and ~1M training samples instead of 256.

Concretely, **each pixel is one training example**: input = the pixel's $(x, y)$ coordinate (normalized to $[-1, 1]^2$), target = the pixel's value. We minimize mean squared error between the MLP's output and the target.

Once trained, the MLP gives us a single function we can evaluate at *any* $(x, y)$ — including positions between the original pixels, or at finer spacing than the original grid. The MLP has learned an **infinite-resolution** representation of the field.

In ML, this idea is called a **neural field** or **implicit neural representation (INR)** — the architecture behind NeRF, SIREN, and most modern coordinate-based generative models. In cosmology, the closest classical analog is **spherical harmonic decomposition** of the CMB: $T(\theta, \phi)$ expressed as a sum of basis functions $Y_{\ell m}$ weighted by coefficients $a_{\ell m}$. A neural field is the non-parametric, fully-flexible generalization — instead of choosing a fixed basis, the MLP learns whatever representation MSE prefers.

## 3. Training

We train the MLP on a random subset of pixels (the **train_fraction** from §2) and measure how well it predicts the rest — the surrogate-model demo in action.

Three choices worth naming explicitly:

- **Sampling: random batch per iteration.** Each step samples a fresh `batch_size` random pixels from the training set (with replacement; statistically equivalent to without-replacement at this scale). Some practitioners use the *epoch* framing instead — one full shuffled pass through the training set per epoch — but for very large training sets, counting raw iterations is more natural, and the two approaches converge to the same place.
- **Optimizer: Adam.** Adam maintains a *per-parameter* learning rate that adapts to each parameter's gradient history. Recall §1.1, where vanilla SGD zigzagged across an elongated valley because one global step size couldn't be right for both axes — Adam handles that automatically. It's the de facto default for neural fields, at the cost of storing two extra tensors per parameter (first and second gradient moments).
- **Scheduler: linear decay.** Learning rate decreases as $lr(step) = lr_0 \cdot (1 - \text{step} / n_\text{steps})$, from `lr` at step 0 down to zero at the final step. Big updates early, fine-tuning near the end.

In [ ]:
# --- Knobs ---
train_fraction = 0.1
hidden_width   = 256
hidden_layers  = 4
activation     = "relu"        # "relu" or "sin"
loss_fn        = "mse"         # "mse" or "l1"
batch_size     = 65536         # also reused as the chunk size for val + reconstruction
n_steps        = 3000
lr             = 1e-3
val_every      = 100
# ---

from tqdm import tqdm
from model import MLPConfig, build_mlp
from dataset import load_image_dataset, make_grid_coords

dataset = load_image_dataset(device=device, train_fraction=train_fraction)
print(f"train: {dataset.train_coords.shape[0]:>10,} pixels")
print(f"val:   {dataset.val_coords.shape[0]:>10,} pixels")

config = MLPConfig(activation=activation, hidden_width=hidden_width, hidden_layers=hidden_layers)
model = build_mlp(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model: {n_params:,} params "
      f"(compression: {dataset.train_values.numel() / n_params:.0f}x vs training data)")

criterion     = {"mse": nn.MSELoss(),                "l1": nn.L1Loss()               }[loss_fn]
criterion_sum = {"mse": nn.MSELoss(reduction="sum"), "l1": nn.L1Loss(reduction="sum")}[loss_fn]

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1 - step / n_steps)

n_train = dataset.train_coords.shape[0]
train_losses, val_losses, val_steps = [], [], []
last_val = float("nan")

pbar = tqdm(range(n_steps), desc="Training")
for step in pbar:
    optimizer.zero_grad()
    idx = torch.randint(0, n_train, (batch_size,), device=device)
    pred = model(dataset.train_coords[idx])
    loss = criterion(pred, dataset.train_values[idx])
    loss.backward()
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())

    if (step + 1) % val_every == 0:
        with torch.no_grad():
            val_loss_sum = 0.0
            for start in range(0, dataset.val_coords.shape[0], batch_size):
                end = start + batch_size
                p = model(dataset.val_coords[start:end])
                val_loss_sum += criterion_sum(p, dataset.val_values[start:end]).item()
            last_val = val_loss_sum / dataset.val_coords.shape[0]
        val_losses.append(last_val)
        val_steps.append(step + 1)

    pbar.set_postfix(train=f"{loss.item():.4f}",
                     val=f"{last_val:.4f}",
                     lr=f"{scheduler.get_last_lr()[0]:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(train_losses, label="train (per step)", alpha=0.4)
ax.semilogy(val_steps, val_losses, "o-", label=f"val (every {val_every} steps)", color="C1")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title(f"Training curve — {activation}-MLP, {loss_fn.upper()} loss, "
             f"{n_params:,} params, {train_fraction*100:.0f}% train")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the trained model at every pixel and reshape to the image grid.
# Chunked in batch_size to stay within GPU memory.
full_coords = make_grid_coords(dataset.image_shape, device=device)
with torch.no_grad():
    chunks = []
    for start in tqdm(range(0, full_coords.shape[0], batch_size), desc="Reconstructing"):
        chunks.append(model(full_coords[start:start + batch_size]))
    predicted = torch.cat(chunks).reshape(dataset.image_shape).cpu().numpy()

# Reconstruct truth in the same normalized space the MLP was trained on
# (min-max to [-1, 1]) so we can compare apples to apples.
truth = data.astype(np.float32)
truth = 2 * (truth - truth.min()) / (truth.max() - truth.min()) - 1

# Bin pixels by truth value and average the signed error within each bin.
# Shows systematic bias as a function of f(x): a flat line at zero means
# the model is unbiased everywhere; a slope or curve means certain truth
# values are systematically over- or under-predicted.
err = (predicted - truth).flatten()
n_bins = 80
counts,  bin_edges = np.histogram(truth.flatten(), bins=n_bins)
err_sum, _         = np.histogram(truth.flatten(), bins=bin_edges, weights=err)
mean_err = np.where(counts > 0, err_sum / np.maximum(counts, 1), np.nan)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(truth,     origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Truth (CMB-like field)")
axes[0].axis("off")

axes[1].imshow(predicted, origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title(f"MLP prediction ({activation})")
axes[1].axis("off")

axes[2].plot(bin_centers, mean_err, "-", color="C0")
axes[2].axhline(0, color="gray", lw=0.5, alpha=0.6)
axes[2].set_xlabel("truth  f(x)")
axes[2].set_ylabel("mean error in bin")
axes[2].set_title("Mean error vs truth value")

plt.tight_layout()
plt.show()